In [1]:
# import os

# os.environ["CUDA_VISIBLE_DEVICES"]="cuda:0"

In [2]:
import torch

torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
if(torch.cuda.device_count() > 1):
    device = 'cuda:0'    

print(torch.__version__)

2.6.0+cu124


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import math
import torch.nn.functional as F

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)


# model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device, dtype=torch.float16)
tokenizer.pad_token = tokenizer.eos_token
special_tokens_dict = {'additional_special_tokens': ['Q: ', 'A: ']}
tokenizer.add_special_tokens(special_tokens_dict) # special token should be added at the end
model.resize_token_embeddings(len(tokenizer))
# model = model.to(device)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(128258, 4096)

In [5]:
# output a list of average logprob, the first is the query itself
def cal_part_logprob_in_batch(_input_text: list, _tokenizer, _device, _max_length=2048):
    with torch.no_grad():
        inputs = _tokenizer(_input_text, return_tensors="pt", padding=True, truncation=True, max_length=_max_length).to(_device)
        outputs = model(**inputs, labels=inputs["input_ids"], loss_type='ForCausalLMLoss')
    
        logits = outputs.logits.to('cpu')
        del outputs
        inputs = inputs.to('cpu')
        attention_mask = inputs["attention_mask"]
        
        input_ids = inputs["input_ids"]
        print(input_ids.shape)

        attention_mask = torch.stack([attention_mask[0]] + [((1-attention_mask[0]) & _m) for _m in attention_mask[1:]]) # mask the query tokens for the later ones
        
        shift_logits = logits[: , :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        
        log_probs = F.log_softmax(shift_logits, dim=-1)
        # print(log_probs)
        log_probs_for_tokens = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
        actual_log_probs_for_tokens = attention_mask[:, 1:] * log_probs_for_tokens  # the first position is the beginning token
        
        actual_nums = attention_mask[:, 1:].sum(dim=1)
        actual_sums = actual_log_probs_for_tokens.sum(dim=1)
        avg_logProbs = actual_sums/actual_nums
    torch.cuda.empty_cache()
    return avg_logProbs.tolist(), log_probs_for_tokens[1:], attention_mask[1:, 1:], input_ids[1:]

##### Idea: calcuate the average logprob per token in a sentence, passage, or a multi-passage passage, it seems not difficult to calculate

In [6]:
import sys
import os
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
analysis_root = os.path.abspath(os.path.join(notebook_dir, '..', 'analysis'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if analysis_root not in sys.path:
    sys.path.insert(0, analysis_root)

from analysis.tools import coherence_cal

In [7]:
torch.cuda.empty_cache()

In [8]:
_ret = 'e5'
res, doc_dict, _ = coherence_cal.get_res_and_dicts('dl', _ret)

loading doc dict
calculating document lengths
~Loading it from the cache
Filename: /mnt/primary/utility_prediction/analysis/tools/coherence_cal.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    55   1047.8 MiB   1047.8 MiB           1   @profile
    56                                         def get_res_and_dicts(_task, _ret):
    57   1047.8 MiB      0.0 MiB           1       material_path = '../../rag_utility'
    58   1047.8 MiB      0.0 MiB           1       splitter = SentenceSplitter(language='en')
    59                                         
    60   1047.8 MiB      0.0 MiB           1       print('loading doc dict')
    61   1047.8 MiB      0.0 MiB           1       if(_task == 'dl'):
    62   1047.8 MiB      0.0 MiB           1           f = open(f'{material_path}/doc_dicts/msmarco_passage_dict.pkl', 'rb')
    63   1052.8 MiB      5.0 MiB           1           dl_19_res = pd.read_csv(f'{material_path}/res/{_ret}_dl_19.csv')
    64   1054.8 MiB      2.0 M

In [9]:
import json
from tqdm import tqdm

_k = 5

prob_res_short = {}
prob_res_long = {}

for qid in tqdm(res.qid.unique()[:1]):
    query_text = res[res.qid==qid]['query'].values[0]
    short_preamble = f'Q: {query_text}\nA: '

    doc_texts_full = res[(res.qid==qid)&(res['rank']<_k)].docno.apply(lambda x: doc_dict[str(x)]).tolist()
    full_context = ''
    for _t, _context_i in zip(doc_texts_full, range(_k)):
        full_context += f'Doc {_context_i}: {_t}\n'
    long_preamble = full_context + short_preamble
    
    _res_per_q = {}

    _batch_size = min(_k, 5) # the number of estimations done at one time
    _i = 0
    _i_long = 0
    while _i < _k:
        doc_texts = res[(res.qid==qid)&(res['rank']>=_i)&(res['rank']<_i+_batch_size)].docno.apply(lambda x: doc_dict[str(x)]).tolist()
        
        doc_texts_with_pre = [short_preamble]# the first is the preamble itself, for calculating the mask
        doc_texts_with_pre += [f'{short_preamble}{_t}' for _t in doc_texts]

        doc_texts_with_context = [long_preamble]
        doc_texts_with_context += [f'{long_preamble}{_t}' for _t in doc_texts]
        # print(doc_texts_with_context)

        # estimate logprob for short preamble
        _res_per_q = {}
        avg_logProb_list, exact_probs_short, mask_short, inputs_short = cal_part_logprob_in_batch(doc_texts_with_pre, tokenizer, device)
        torch.cuda.empty_cache()
        _res_per_q.update(dict(zip(range(_i, _i+len(doc_texts_with_pre)-1), zip(avg_logProb_list[1:], exact_probs_short, mask_short, inputs_short)))) # discard the first that is about preamble
        _i += _batch_size

        # estimate logprob for long preamble
        _res_per_q_long = {}
        avg_logProb_list, exact_probs_long, mask_long, inputs_long = cal_part_logprob_in_batch(doc_texts_with_context, tokenizer, device)
        torch.cuda.empty_cache()
        _res_per_q_long.update(dict(zip(range(_i_long, _i_long+len(doc_texts_with_context)-1), zip(avg_logProb_list[1:], exact_probs_long, mask_long, inputs_long)))) # discard the first that is about preamble
        _i_long += _batch_size

    prob_res_short.update({qid: _res_per_q})
    prob_res_long.update({qid: _res_per_q_long})


  0%|          | 0/1 [00:00<?, ?it/s]

torch.Size([6, 76])
torch.Size([6, 411])


100%|██████████| 1/1 [00:01<00:00,  1.93s/it]


In [14]:
prob_res_short

{'156493': {0: (-2.986328125,
   tensor([-1.6422e+01, -2.4656e+01, -1.4188e+01, -5.7227e+00, -8.6406e+00,
           -2.5508e+00, -1.8047e+01, -1.3555e+01, -1.3580e-02, -1.2930e+01,
           -1.4609e+00, -1.0928e+00, -6.8516e+00, -3.9246e-02, -3.2202e-01,
           -9.3201e-02, -5.4062e+00, -7.7820e-03, -3.2246e+00, -6.8867e+00,
           -1.0459e+00, -2.9375e+00, -1.0766e+01, -3.8986e-03, -6.0638e-02,
           -3.1641e-01, -2.4475e-01, -3.4336e+00, -2.8296e-01, -1.0730e-01,
           -7.1143e-01, -3.8770e-01, -3.7832e+00, -1.3211e+01, -5.6030e-02,
           -2.1255e-02, -9.2480e-01, -9.0078e+00, -1.8809e+00, -4.6367e+00,
           -6.0820e+00, -3.7384e-02, -4.7891e+00, -6.6309e-01, -1.7051e+00,
           -3.4551e+00, -1.5503e-02, -3.4229e-01, -5.3516e+00, -6.9727e+00,
           -2.0581e-01, -2.9259e-03, -2.3242e-01, -2.7930e-01,  0.0000e+00,
           -4.4648e+00, -4.0312e+00, -2.8281e+00, -3.8986e-03, -2.8633e+00,
           -7.6914e+00, -1.1523e+01, -3.1660e+00, -5.1016e

In [13]:
# short_seq

check_qid = '156493'
check_doc_rank = 4

for_this_doc = prob_res_short[check_qid][check_doc_rank]
# print(tokenizer.decode(for_this_doc[3][1:]))
# print(tokenizer.decode(for_this_doc[3][1:][for_this_doc[2].bool()]))
short_seq = for_this_doc[1][for_this_doc[2].bool()]

# long_seq

for_this_doc = prob_res_long[check_qid][check_doc_rank]
# print(tokenizer.decode(for_this_doc[3][1:]))
long_seq = for_this_doc[1][for_this_doc[2].bool()]

lsd = long_seq - short_seq
lsd_mask = lsd>=1.5

print(short_seq.mean())

tensor(-2.9375, dtype=torch.float16)


In [32]:
for_this_doc[3][1:][for_this_doc[2].bool()]

tensor([   45, 78206, 32293,   304, 28415,   578, 48078, 32293,   304, 28415,
         1754,   374,   264, 45370, 10292,  2728,   555,   279, 16591, 31209,
        16192,   315, 23199,   369,  1884,   889,   617,  1903,   279,  1455,
        19310, 19564,   369, 43384,   304,   279,  2115,   315, 22027,    13,
         1102,   374,   832,   315,   279,  4330, 48078,  2394,  4861,  9749,
          555,   279,   690,   315, 42592, 48078,   304,   220,  9378,    20,
          323, 22034,  2533,   220,  7028,    16,    26,   279,  3885,  1694,
          279, 48078, 32293,   304, 42846,    11, 48078, 32293,   304, 47470,
           11, 48078, 26888, 32293,    11,   323, 48078, 32293,   304, 95946,
          477, 19152,    13,   578,  1176, 48078, 32293,   304, 28415,   574,
        22034,   311, 83323, 93537,   432,  3029,   406,  4469,   304, 18324,
          315,   279, 24674,  3600,   568])

In [33]:
tokenizer.decode(for_this_doc[3][1:][for_this_doc[2].bool()])

'Nobel Prize in Physics The Nobel Prize in Physics () is a yearly award given by the Royal Swedish Academy of Sciences for those who have made the most outstanding contributions for mankind in the field of physics. It is one of the five Nobel Prizes established by the will of Alfred Nobel in 1895 and awarded since 1901; the others being the Nobel Prize in Chemistry, Nobel Prize in Literature, Nobel Peace Prize, and Nobel Prize in Physiology or Medicine. The first Nobel Prize in Physics was awarded to physicist Wilhelm Röntgen in recognition of the extraordinary services he'

In [34]:
# tokenizer.convert_ids_to_tokens(for_this_doc[3][1:][for_this_doc[2].bool()][lsd_mask])
tokenizer.decode(for_this_doc[3][1:][for_this_doc[2].bool()][lsd_mask])

'N The () yearly for those outstanding It in and awarded; Peace first to physicist R recognition the'

In [35]:
tokenizer.decode(for_this_doc[3][1:][for_this_doc[2].bool()][short_seq>-0.0001])

' Academy of Sciences most51901 Prize or'

In [36]:
tokenizer.decode(for_this_doc[3][1:][for_this_doc[2].bool()][long_seq>-0.0001])

' yearly who mostizes18951901 Prize Nobel Prize Prizegen'

In [37]:
tokenizer.decode(for_this_doc[3][1:])

'Doc 0: receive a diploma, a medal and a document confirming the prize amount. Nobel Prize in Physics The Nobel Prize in Physics () is a yearly award given by the Royal Swedish Academy of Sciences for those who have made the most outstanding contributions for mankind in the field of physics. It is one of the five Nobel Prizes established by the will of Alfred Nobel in 1895 and awarded since 1901; the others being the Nobel Prize in Chemistry, Nobel Prize in Literature, Nobel Peace Prize, and Nobel Prize in Physiology or Medicine. The first Nobel Prize in Physics was\nDoc 1: Nobel Prize in Physics The Nobel Prize in Physics () is a yearly award given by the Royal Swedish Academy of Sciences for those who have made the most outstanding contributions for mankind in the field of physics. It is one of the five Nobel Prizes established by the will of Alfred Nobel in 1895 and awarded since 1901; the others being the Nobel Prize in Chemistry, Nobel Prize in Literature, Nobel Peace Prize, and N

In [38]:
res[(res.qid==check_qid)&(res['rank']==check_doc_rank)].docno.apply(lambda x: doc_dict[str(x)]).values[0]

'Nobel Prize in Physics The Nobel Prize in Physics () is a yearly award given by the Royal Swedish Academy of Sciences for those who have made the most outstanding contributions for mankind in the field of physics. It is one of the five Nobel Prizes established by the will of Alfred Nobel in 1895 and awarded since 1901; the others being the Nobel Prize in Chemistry, Nobel Prize in Literature, Nobel Peace Prize, and Nobel Prize in Physiology or Medicine. The first Nobel Prize in Physics was awarded to physicist Wilhelm Röntgen in recognition of the extraordinary services he'

In [39]:
colon_tokens = [tok for tok in tokenizer.get_vocab().keys() if ":" in tok]

print(f"Found {len(colon_tokens)} tokens containing ':'")
print(colon_tokens[:50])  # show first 50 as preview

Found 602 tokens containing ':'
[':";čĊ', ':^', ':id', '(?:', 'Ġ:";Ċ', ':get', '":ĊĊ', ':focus', ':false', ']):čĊ', "']):Ċ", 'Ġ::Ċ', 'Ġ:Ċ', 'âĢĿ:', '::::', "(':')[", 'Ġ:+:', ':key', '[:,:,', '[:,', '?:', '*>::', ':flex', 'Ġ"":Ċ', ':center', ':j', '::::::::', ":''", ':c', ':frame', ':.:.:', ':]Ċ', '*:', ':CGRect', ":'+", "__':Ċ", '":"\'', 'Ġ-:-', 'Ġ:",', "Ġ'::", 'Ġ):', 'Ġ:=', '():ĊĊ', ':/', '#:', ':Get', '.:', ':[[', 'Ġ:\\', ':YES']


In [29]:
tokenizer('A:Doctopus')['input_ids']

[128000, 32, 25, 5519, 302, 46970]

In [30]:
tokenizer.decode([25])

':'

In [52]:
import json
from tqdm import tqdm

_k = 8

prob_res_short = {}
prob_res_long = {}

for qid in tqdm(res.qid.unique()[:1]):
    query_text = res[res.qid==qid]['query'].values[0]
    short_preamble = f'Q: {query_text}\nA: '
    
    _res_per_q = {}

    _batch_size = min(_k, 5) # the number of estimations done at one time
    _i = 0
    while _i < _k:
        doc_texts = res[(res.qid==qid)&(res['rank']>=_i)&(res['rank']<_i+_batch_size)].docno.apply(lambda x: doc_dict[str(x)]).tolist()
        
        doc_texts_with_pre = [short_preamble]# the first is the preamble itself, for calculating the mask
        doc_texts_with_pre += [f'{short_preamble}{_t}' for _t in doc_texts]

        # estimate logprob for short preamble
        # _res_per_q = {}
        avg_logProb_list, exact_probs_short, mask_short, inputs_short = cal_part_logprob_in_batch(doc_texts_with_pre, tokenizer, device)
        torch.cuda.empty_cache()
        _res_per_q.update(dict(zip(range(_i, _i+len(doc_texts_with_pre)-1), zip(avg_logProb_list[1:])))) # discard the first that is about preamble
        _i += _batch_size

    prob_res_short.update({qid: _res_per_q})

  0%|          | 0/1 [00:00<?, ?it/s]

torch.Size([6, 141])


100%|██████████| 1/1 [00:01<00:00,  1.45s/it]

torch.Size([6, 145])


In [53]:
prob_res_short

{'test_0': {0: (-1.2568359375,),
  1: (-1.044921875,),
  2: (-1.9873046875,),
  3: (-3.181640625,),
  4: (-2.30078125,),
  5: (-2.373046875,),
  6: (-1.869140625,),
  7: (-1.283203125,),
  8: (-1.9736328125,),
  9: (-2.068359375,)}}